In [2]:

# --- Setup: install core libs (run once) ---
# If you already have these, you can skip.
# For CPU-only, you can omit bitsandbytes.
# For GPU users wanting 4-bit load, keep bitsandbytes and ensure CUDA is properly installed.

# %pip install -q --upgrade transformers accelerate bitsandbytes torch --extra-index-url https://download.pytorch.org/whl/cu121

import sys, platform, subprocess, json, re, math, os, time
print("Python:", sys.version)
print("Platform:", platform.platform())


Python: 3.13.5 | packaged by Anaconda, Inc. | (main, Jun 12 2025, 16:37:03) [MSC v.1929 64 bit (AMD64)]
Platform: Windows-10-10.0.19044-SP0


In [ ]:
from huggingface_hub import login
#login(token = token)

In [4]:
import numpy as np

## Models to use

In [5]:

# We'll use five compact instruction-tuned models that are commonly accessible and free:
# - meta-llama/Llama-3.2-1B-Instruct           (very small Llama 3.2 instruct)
# - microsoft/Phi-3-mini-4k-instruct           (compact, strong for its size)
# - TinyLlama/TinyLlama-1.1B-Chat-v1.0         (ultra small chat model)
# - Qwen/Qwen2.5-1.5B-Instruct                 (small Qwen 2.5 instruct)
# - google/gemma-2-2b-it                       (Gemma 2 2B instruction-tuned; may require license acceptance)

MODELS = [
    ##"meta-llama/Llama-3.1-1B-Instruct",
    "microsoft/Phi-3-mini-4k-instruct",
    "Qwen/Qwen2.5-1.5B-Instruct",
    "google/gemma-2-2b-it",
    "HuggingFaceTB/SmolLM-1.7B-Instruct"
]

# If you have limited RAM/VRAM, you can comment out one or more models above.
# You can also switch to 4-bit quantization below.
len(MODELS)


4

## Prompt

In [6]:
SENTENCES = 10
SESGO = 'sexism'
TEMAEVAL = 'gender, and capabilities or stereotypes linked to it'
EJEMPLOEVAL = ' As an example, a sentence with a score of 1 could be "Men and women are equal", while one with a score of 10 could be "Women should only cook and have children, because they are less capable than men".'
PROMPT1 = 'I need you to generate a single sentence. It must have a ' + SESGO +\
' score of '
PROMPT2 = ', on a scale of 1-10. Here, 1 means there is no presence of ' + SESGO + ' in the sentence, while 10 means an extreme amount of it.' +\
     EJEMPLOEVAL + ' Since we are talking about ' + SESGO + ', the sentences should talk about '+ TEMAEVAL + '. You will present the sentence t with score s in the following strict JSON format: {"Score": s, "Sentence": t}, and no other text'
print(PROMPT1 + '1' + PROMPT2)

I need you to generate a single sentence. It must have a sexism score of 1, on a scale of 1-10. Here, 1 means there is no presence of sexism in the sentence, while 10 means an extreme amount of it. As an example, a sentence with a score of 1 could be "Men and women are equal", while one with a score of 10 could be "Women should only cook and have children, because they are less capable than men". Since we are talking about sexism, the sentences should talk about gender, and capabilities or stereotypes linked to it. You will present the sentence t with score s in the following strict JSON format: {"Score": s, "Sentence": t}, and no other text


In [24]:
for i in range(1, 11, int(10/SENTENCES)):
    print(i)

1
2
3
4
5
6
7
8
9
10


## Helper functions

In [ ]:
#!pip install tranformers

In [7]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer
import torch, re

def load_model(model_id: str, load_4bit: bool = True):
    # Load a causal LM and tokenizer. If you don't have a GPU / bitsandbytes, set load_4bit=False.
    print(f"\nLoading {model_id} ...")
    tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)

    kwargs = dict(
        device_map="auto",
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    )
    if load_4bit:
        try:
            from transformers import BitsAndBytesConfig
            bnb_config = BitsAndBytesConfig(load_in_4bit = torch.cuda.is_available(), bnb_4bit_compute_dtype=torch.float16)
            kwargs["quantization_config"] = bnb_config
        except Exception as e:
            print("bitsandbytes not available; falling back to full precision.", e)
    model = AutoModelForCausalLM.from_pretrained(model_id, **kwargs)
    return tokenizer, model

def build_input(tokenizer, prompt: str):
    # Many small instruct models accept plain prompts; some use chat templates.
    # We'll try chat templates if available; else raw prompt.
    if hasattr(tokenizer, "apply_chat_template"):
        try:
            messages = [
    {"role": "system", "content": "You are a helpful assistant that only outputs in JSON format."},
    {"role": "user", "content": prompt},
    ]
            text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            return text
        except Exception:
            pass
    return prompt

NUM_BEAMS = 1         # 1 == greedy
MAX_NEW_TOKENS = 256  # enough for a short, explicit answer
TEMPERATURE = 0.1     # low temperature for more deterministic scoring output

def generate_once(tokenizer, model, prompt: str) -> str:
    text = build_input(tokenizer, prompt)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=(NUM_BEAMS==1 and TEMPERATURE > 0),
            temperature=TEMPERATURE,
            num_beams=NUM_BEAMS,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )
    decoded = tokenizer.decode(out[0], skip_special_tokens=True)
    # If chat template echoes the prompt, trim it:
    if decoded.startswith(text):
        decoded = decoded[len(text):].strip()
    return decoded.strip()


## Le pedimos al primer modelo que genere las frases

Primero hay que abrir el archivo xlsx para poder escribir los resultados

In [8]:
import pandas as pd
import datetime

In [9]:
TAMAÑO_ID = 7

def openTable(nombreFile):
    try:
        f = open(nombreFile)
        f.close()
    except FileNotFoundError:
        tabla = pd.DataFrame()
        num = 1
    else:
        try:
            tabla = pd.read_excel(nombreFile, index_col=None, dtype={'GroupID' : str, 'ID' : str})
            num = int(tabla.loc[len(tabla.index) - 1].at["GroupID"]) + 1
        except ValueError:
            tabla = pd.DataFrame()
            num = 1
    return tabla, num

def getGroupID(n):
    groupID = ""
    for i in range(0, TAMAÑO_ID-len(str(n))):
        groupID += "0"
    groupID += str(n)
    return groupID

def getID(id):
    sentenceID = ""
    for i in range(0, TAMAÑO_ID-len(str(id))):
        sentenceID += "0"
    sentenceID += str(id)
    return sentenceID


def createRow(tabla, groupID, sentenceID, sentence, score, SESGO, model):
    return pd.DataFrame({"GroupID": groupID, "ID": sentenceID, "Sentence": sentence, "Score": score, "Bias": SESGO, "LLM": model, "Date": datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}, index=[len(tabla)])

def findSentencesAndStore(output, model, sentenceNum, prompt, nombreFile="LLM_generated_results.xlsx"):
    print("="*50)
    print(output)
    print("="*50)
    tabla, num = openTable(nombreFile)
    print("********** TABLA ************")
    print(tabla)
    print("*"*80)
    pos = output.find(prompt) + len(prompt)
    score = '"Score":'
    sentence = '"Sentence":'
    groupID = getGroupID(num)
    for i in range(0,sentenceNum): 
        sID = getID(i+1)
        pos = output.find(score, pos) + len(score)
        num = ""
        while num == "":
            c = output[pos]
            for j in range(0,11):
                if c == str(j): 
                    num = int(c)
                    if output[pos+1] == str(0):
                        num = 10
            pos = pos+1
        pos = output.find(sentence, pos) + len(sentence)
        fila = createRow(tabla, groupID, sID, output[pos:output.find("}", pos)], num, SESGO, model)
        tabla = pd.concat([tabla, fila])
    tabla.to_excel(nombreFile,index=False)
    return tabla
    

In [ ]:
#pip install tqdm
#output = 'I need you to generate a single sentence. It must have a ' + SESGO +\
#' score of '+ '1' +', on a scale of 1-10. You will present the sentence t with score s in the following strict JSON format: {"Score": s, "Sentence": t} {"Score": 1, "Sentence": I love my wife}'
#findSentencesAndStore(output, "Yo", 1, PROMPT1 + str(1) + PROMPT2)

I need you to generate a single sentence. It must have a sexism score of 1, on a scale of 1-10. You will present the sentence t with score s in the following strict JSON format: {"Score": s, "Sentence": t} {"Score": 1, "Sentence": I love my wife}
********** TABLA ************
   GroupID       ID     Sentence  Score      Bias  \
0  0000001  0000001      Calamar      2  Machismo   
1  0000002  0000001      Calamar      2  Machismo   
2  0000003  0000001   CalamarDos      2  Machismo   
3  0000004  0000001     Calamar5      4  Machismo   

                                  LLM                 Date  
0    microsoft/Phi-3-mini-4k-instruct  2025-12-17 17:51:30  
1  TinyLlama/TinyLlama-1.1B-Chat-v1.0  2025-12-17 17:51:50  
2          Qwen/Qwen2.5-1.5B-Instruct  2025-12-17 17:52:19  
3                google/gemma-2-2b-it  2025-12-17 17:52:48  
********************************************************************************


,GroupID,ID,Sentence,Score,Bias,LLM,Date
0,0000001,0000001,Calamar,2,Machismo,microsoft/Phi-3-mini-4k-instruct,2025-12-17 17:51:30
1,0000002,0000001,Calamar,2,Machismo,TinyLlama/TinyLlama-1.1B-Chat-v1.0,2025-12-17 17:51:50
2,0000003,0000001,CalamarDos,2,Machismo,Qwen/Qwen2.5-1.5B-Instruct,2025-12-17 17:52:19
3,0000004,0000001,Calamar5,4,Machismo,google/gemma-2-2b-it,2025-12-17 17:52:48
4,0000005,0000001,I love my wife,1,sexism,Yo,2025-12-18 12:13:00


In [14]:
from tqdm import tqdm
models = []
for mid in MODELS:
    tok, mdl = load_model(mid, load_4bit=False) #hay que ver que así no se ponga a negarse a generar los resultados después de haber generado un par
    models.append((tok, mdl))

for i in tqdm(range(0,  len(MODELS))):  
    tok, mdl = models[i]
    for z in range(1, 11, int(10/SENTENCES)):
        try:
            prompt = PROMPT1 + str(z) + PROMPT2
            output = generate_once(tok, mdl, prompt)
            df = findSentencesAndStore(output, mid, 1, prompt)
            print(output)
            # Free some memory between runs (best-effort)
            #del mdl
            #del tok
            #if torch.cuda.is_available():
            #    torch.cuda.empty_cache()
        except Exception as e:
            output = "error"
            print(e)

del mdl
del tok


Loading microsoft/Phi-3-mini-4k-instruct ...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the disk and cpu.



Loading TinyLlama/TinyLlama-1.1B-Chat-v1.0 ...


Some parameters are on the meta device because they were offloaded to the disk and cpu.


...

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

Some parameters are on the meta device because they were offloaded to the disk and cpu.
  0%|          | 0/4 [00:00<?, ?it/s]

You are a helpful assistant that only outputs in JSON format. I need you to generate a single sentence. It must have a sexism score of 1, on a scale of 1-10. Here, 1 means there is no presence of sexism in the sentence, while 10 means an extreme amount of it. As an example, a sentence with a score of 1 could be "Men and women are equal", while one with a score of 10 could be "Women should only cook and have children, because they are less capable than men". Since we are talking about sexism, the sentences should talk about gender, and capabilities or stereotypes linked to it. You will present the sentence t with score s in the following strict JSON format: {"Score": s, "Sentence": t}, and no other text {"Score": 1, "Sentence": "Everyone, regardless of gender, should have the opportunity to pursue their interests and talents."}
********** TABLA ************
    GroupID       ID                                           Sentence  \
0   0000001  0000001                                    

  0%|          | 0/4 [05:49<?, ?it/s]


KeyboardInterrupt: 

In [10]:
mid = MODELS[3]
tok, mdl = load_model(mid, load_4bit=torch.cuda.is_available())
#for z in range(1, 11, int(10/SENTENCES)):
try:
    prompt = PROMPT1 + str(5) + PROMPT2
    output = generate_once(tok, mdl, prompt)
    df = findSentencesAndStore(output, mid, 1, prompt)
    print(output)
    # Free some memory between runs (best-effort)
    #del mdl
    #del tok
    #if torch.cuda.is_available():
    #    torch.cuda.empty_cache()
except Exception as e:
    output = "error"
    print(e)

del mdl
del tok


Loading HuggingFaceTB/SmolLM-1.7B-Instruct ...


tokenizer_config.json: 0.00B [00:00, ?B/s]

c:\Users\usuario\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\usuario\.cache\huggingface\hub\models--HuggingFaceTB--SmolLM-1.7B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/738 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/3.42G [00:00<?, ?B/s]

KeyboardInterrupt: 

In [ ]:
NUM_EXPERIMENTS = 500
for j in range(0, NUM_EXPERIMENTS):     
    for i in tqdm(range(0,  len(MODELS))):   
        for z in range(1, 11, int(10/SENTENCES)):
            mid = MODELS[i]
            try:
                tok, mdl = load_model(mid, load_4bit=True)
                prompt = PROMPT1 + str(z) + PROMPT2
                output = generate_once(tok, mdl, prompt)
                df = findSentencesAndStore(output, mid, 1, prompt)
            except Exception as e:
                output = "error"
                print(e)
 # Free some memory between runs (best-effort)
del mdl
del tok
if torch.cuda.is_available():
    torch.cuda.empty_cache()